# Fine-tuning results — visualization

This notebook only *visualizes* results that are already computed by the pipeline —
it does not train or generate anything itself. Training runs via `mlx_lm.lora`
(long-running, Metal-backed, CLI-driven — see `fine_tuning/lora_config.yaml`) and
generation runs via `fine_tuning/generate_predictions.py`. Both are better suited
to a terminal/background process than a notebook cell: a 200-iteration LoRA run
takes long enough that a blocking notebook cell adds fragility (kernel
disconnects losing the run) without adding anything MLX-LM's own
iteration/loss logging doesn't already give you.

What this notebook adds is exactly what `rag/mongodb_nl_to_sql_rag.ipynb`'s
cells 10-12 already do for the RAG arm: load the already-computed result files
and produce the comparison charts, saved to `fine_tuning/outputs/figures/` —
same convention as `outputs/figures/` (baseline) and `rag/outputs/figures/` (RAG),
so all three stages keep the same look and the same place to find their charts.

In [1]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Robust root-detection -- same pattern as rag/mongodb_nl_to_sql_rag.ipynb's
# viz cells, works whether this notebook runs from the repo root or from
# inside fine_tuning/ (this notebook's own directory).
ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / "data").exists() and (ROOT / "rag").exists():
        break
    ROOT = ROOT.parent

FT_FIGURES = ROOT / "fine_tuning" / "outputs" / "figures"
FT_FIGURES.mkdir(parents=True, exist_ok=True)

In [2]:
# ---------------------------------------------------
# Load data -- reads:
#   data/finetuned_execution_results.json   (fine_tuning/generate_predictions.py
#                                             -> normalize.py -> evaluation/execute_queries.py)
#   rag/outputs/rag_vs_baseline_scores.csv   (rag/score_rag.py, K=10 run --
#                                             same file the RAG notebook itself reads)
# Fail loud if either is missing rather than silently plotting stale numbers.
# ---------------------------------------------------
ft_results_path = ROOT / "data" / "finetuned_execution_results.json"
rag_scores_path = ROOT / "rag" / "outputs" / "rag_vs_baseline_scores.csv"

if not ft_results_path.exists():
    raise FileNotFoundError(f"{ft_results_path} missing -- run the fine-tuning eval pipeline first")
if not rag_scores_path.exists():
    raise FileNotFoundError(f"{rag_scores_path} missing -- run rag/score_rag.py first")

ft_results = json.load(open(ft_results_path))
rag_scores = pd.read_csv(rag_scores_path)

baseline_row = rag_scores[rag_scores["Arm"] == "Qwen-Baseline(test-slice)"].iloc[0]
rag_row = rag_scores[rag_scores["Arm"] == "Qwen-RAG"].iloc[0]

ft_total = len(ft_results)
ft_correct = sum(1 for r in ft_results if r["execution_accuracy"])
ft_accuracy = 100 * ft_correct / ft_total

print(f"Baseline (test-slice): {baseline_row['Correct']}/{int(baseline_row['Total'])} ({baseline_row['Accuracy']:.1f}%)")
print(f"RAG (K=10):            {rag_row['Correct']}/{int(rag_row['Total'])} ({rag_row['Accuracy']:.1f}%)")
print(f"Fine-tuned (LoRA):     {ft_correct}/{ft_total} ({ft_accuracy:.1f}%)")

Baseline (test-slice): 3/61 (4.9%)
RAG (K=10):            15/61 (24.6%)
Fine-tuned (LoRA):     20/61 (32.8%)


In [3]:
# ---------------------------------------------------
# Figure 1: Execution Accuracy -- Baseline vs RAG vs Fine-tuned
# All three scored on the identical 61-case held-out slice, same gold, same
# scoring code (evaluation/execute_queries.py) -- so this is a fair 3-arm
# comparison, not three different test sets.
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
labels = ["Qwen-Baseline\n(test-slice)", "Qwen-RAG\n(K=10)", "Qwen-Fine-tuned\n(LoRA)"]
vals = [baseline_row["Accuracy"], rag_row["Accuracy"], ft_accuracy]
counts = [(baseline_row["Correct"], baseline_row["Total"]), (rag_row["Correct"], rag_row["Total"]), (ft_correct, ft_total)]
bars = plt.bar(labels, vals, color=["#888888", "#2a7de1", "#2eb872"])
plt.ylabel("Execution Accuracy (%)")
plt.title(f"Baseline vs RAG vs Fine-tuned Execution Accuracy (n={ft_total})")
plt.ylim(0, 100)
for bar, v, (c, t) in zip(bars, vals, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
        f"{v:.1f}%\n({int(c)}/{int(t)})",
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plt.savefig(FT_FIGURES / "finetuned_vs_all_arms.png", dpi=300)
plt.close()
print(f"Saved {FT_FIGURES / 'finetuned_vs_all_arms.png'}")

Saved /mnt/user-data/uploads/CSAIML-Capstone-Project-20/fine_tuning/outputs/figures/finetuned_vs_all_arms.png


In [4]:
# ---------------------------------------------------
# Figure 2: Fine-tuned outcome breakdown -- where the 41 non-matching cases
# actually land. Mirrors the RAG funnel diagnostic's spirit (show WHERE
# the pipeline is losing points, not just the final number).
# ---------------------------------------------------
status_counts = Counter(r["status"] for r in ft_results)
hard_fail = status_counts.get("FAIL", 0)
ran_ok = status_counts.get("PASS", 0)
wrong_nonempty = sum(1 for r in ft_results if r["status"] == "PASS" and r.get("non_empty_rate") and not r["execution_accuracy"])
empty_but_ran = sum(1 for r in ft_results if r["status"] == "PASS" and not r.get("non_empty_rate"))
assert ft_correct + wrong_nonempty + empty_but_ran + hard_fail == ft_total, "breakdown doesn't sum to total -- check status/field assumptions"

plt.figure(figsize=(6.5, 4))
stages = ["Correct\n(exec. accuracy)", "Ran, non-empty\nbut wrong", "Ran, empty\nresult", "Hard failure\n(syntax/safety/error)"]
outcome_counts = [ft_correct, wrong_nonempty, empty_but_ran, hard_fail]
colors = ["#2eb872", "#f0a500", "#c98a2c", "#c0392b"]
bars = plt.bar(stages, outcome_counts, color=colors)
plt.ylabel(f"Number of cases (of {ft_total})")
plt.title(f"Fine-tuned Outcome Breakdown, {ft_total}-case Holdout")
plt.ylim(0, max(outcome_counts) + 5)
for bar, c in zip(bars, outcome_counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.6,
        f"{c}/{ft_total}\n({c/ft_total*100:.1f}%)",
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plt.savefig(FT_FIGURES / "finetuned_outcome_breakdown.png", dpi=300)
plt.close()
print(f"Saved {FT_FIGURES / 'finetuned_outcome_breakdown.png'}")

Saved /mnt/user-data/uploads/CSAIML-Capstone-Project-20/fine_tuning/outputs/figures/finetuned_outcome_breakdown.png


## Figure 3 — training loss curve (partial: iterations 100-200 only)

`mlx_lm.lora`'s own per-iteration loss/timing log was never redirected to a
file during the real training run -- it only printed to the terminal, and by
the time this was noticed, terminal scrollback only still had the tail end
of the run. The snippet that survived covers **iterations 100-200 out of
200** (the second half), pasted verbatim into
`fine_tuning/training_log_snippet_iters100-200.txt` and parsed mechanically
below with regex -- not hand-transcribed -- so the plotted numbers are
exactly what `mlx_lm.lora` printed, with no manual copying error possible.

**What this figure can and can't show**: it shows the loss was still
falling through the back half of training (train loss ~0.087 -> 0.055, val
loss 0.089 -> 0.059 from iter 100 to iter 200) -- there's no sign it had
flattened out by iteration 200, which argues against having overtrained.
It **cannot** show the shape of the first half (iterations 1-99), which is
where the largest drop in a loss curve typically happens -- that portion of
the log is gone and is not reconstructed or estimated here.

**Right-sizing note**: mechanically parsing the surviving raw log text
(instead of hand-typing the numbers into a list) matches the rigor already
established elsewhere in this project (e.g. the mechanical failure-mode
taxonomy in `visualize_cross_arm_comparison.ipynb`) at zero extra cost --
there's no reason to introduce transcription risk when the exact source
text is sitting right there. Reconstructing the missing first-half numbers
(e.g. by interpolating from the final adapter alone) would not be
right-sized -- it would be presenting a guess as measured data, which this
project's other findings have specifically avoided doing throughout
(see the fp16 non-determinism writeup and the RAG accuracy range in the
project status for two examples of the same principle).

In [5]:
# ---------------------------------------------------
# Parse fine_tuning/training_log_snippet_iters100-200.txt mechanically via
# regex -- the raw text is mlx_lm.lora's own stdout, pasted verbatim, not
# retyped. Fail loud if the file is missing or a line-shape assumption
# breaks, rather than silently plotting an empty/partial chart.
# ---------------------------------------------------
import re

log_path = ROOT / "fine_tuning" / "training_log_snippet_iters100-200.txt"
if not log_path.exists():
    raise FileNotFoundError(f"{log_path} missing -- this is the pasted mlx_lm.lora console snippet")

log_text = log_path.read_text()

train_re = re.compile(
    r"Iter (\d+): Train loss ([\d.]+), Learning Rate ([\d.eE+-]+), "
    r"It/sec ([\d.]+), Tokens/sec ([\d.]+), Trained Tokens (\d+), Peak mem ([\d.]+) GB"
)
val_re = re.compile(r"Iter (\d+): Val loss ([\d.]+), Val took ([\d.]+)s")

train_rows = [
    {
        "iter": int(m.group(1)), "train_loss": float(m.group(2)),
        "lr": float(m.group(3)), "it_per_sec": float(m.group(4)),
        "tokens_per_sec": float(m.group(5)), "trained_tokens": int(m.group(6)),
        "peak_mem_gb": float(m.group(7)),
    }
    for m in train_re.finditer(log_text)
]
val_rows = [
    {"iter": int(m.group(1)), "val_loss": float(m.group(2)), "val_took_s": float(m.group(3))}
    for m in val_re.finditer(log_text)
]

assert len(train_rows) == 11, f"expected 11 train-loss lines (iters 100,110,...,200), parsed {len(train_rows)}"
assert len(val_rows) == 6, f"expected 6 val-loss lines (iters 100,120,...,200), parsed {len(val_rows)}"

train_df = pd.DataFrame(train_rows).sort_values("iter")
val_df = pd.DataFrame(val_rows).sort_values("iter")
print(train_df[["iter", "train_loss", "it_per_sec", "tokens_per_sec"]].to_string(index=False))
print()
print(val_df.to_string(index=False))

 iter  train_loss  it_per_sec  tokens_per_sec
  100       0.087       0.888        2094.694
  110       0.083       0.917        2163.789
  120       0.067       0.933        2177.237
  130       0.070       0.888        2099.368
  140       0.061       0.907        2122.507
  150       0.068       0.890        2121.117
  160       0.054       0.883        2072.386
  170       0.057       0.870        2064.663
  180       0.051       0.883        2053.561
  190       0.054       0.860        2030.668
  200       0.055       0.863        2029.857

 iter  val_loss  val_took_s
  100     0.089       5.686
  120     0.078       5.695
  140     0.070       5.756
  160     0.066       5.855
  180     0.062       5.989
  200     0.059       6.121


In [6]:
# ---------------------------------------------------
# Figure 3: Training loss vs iteration, second half only (iters 100-200).
# Plotted on the actual iteration numbers (100, 110, ... 200 for train;
# 100, 120, ... 200 for val) -- not resampled/interpolated -- with a shaded
# note that iterations 1-99 aren't available, so the chart can't be
# mistaken for the full run at a glance.
# ---------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(train_df["iter"], train_df["train_loss"], marker="o", color="#2eb872", label="Train loss", linewidth=2)
ax.plot(val_df["iter"], val_df["val_loss"], marker="s", color="#c0392b", label="Val loss", linewidth=2)

ymax = max(train_df["train_loss"].max(), val_df["val_loss"].max()) * 1.15
ax.set_ylim(0, ymax)
ax.set_xlim(0, 205)
ax.axvspan(0, 100, color="#dddddd", alpha=0.5, zorder=0)
ax.text(50, ymax * 0.92, "iters 1-99\nlog not available", ha="center", va="top", fontsize=9, color="#666666", style="italic")

ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("LoRA Fine-tuning Loss vs. Iteration\n(measured second half, iters 100–200 of 200)")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FT_FIGURES / "finetuned_loss_curve.png", dpi=300)
plt.close()
print(f"Saved {FT_FIGURES / 'finetuned_loss_curve.png'}")

Saved /mnt/user-data/uploads/CSAIML-Capstone-Project-20/fine_tuning/outputs/figures/finetuned_loss_curve.png


In [7]:
# ---------------------------------------------------
# Training time: what's measured vs. what's estimated.
#
# MEASURED: wall-clock for iterations 101-200, reconstructed from the
# logged it/sec of each 10-iteration reporting block (iter 110's it/sec
# covers iters 101-110, iter 120's covers 111-120, etc. -- iter 100's own
# it/sec covers iters 91-100, which is outside this window, so it's
# excluded) plus the five "Val took Xs" pauses that fall in this window
# (at iters 120/140/160/180/200; the iter-100 val pause happened before
# this window and is also excluded).
#
# ESTIMATED (clearly separated, not presented as measured): if the first
# 100 iterations ran at a similar steady-state throughput -- plausible
# since it/sec is already fairly stable (0.86-0.93) across the whole
# visible window, but not verifiable without the missing log lines, and
# NOT accounting for one-time model-load/graph-compile overhead that
# typically front-loads onto the first few iterations -- the full
# 200-iteration run would total roughly double the measured half.
# ---------------------------------------------------
second_half_blocks = train_df[train_df["iter"] > 100]  # iters 110..200, i.e. windows covering 101-200
train_step_seconds = (10 / second_half_blocks["it_per_sec"]).sum()

val_in_window = val_df[val_df["iter"] > 100]  # iters 120..200
val_seconds = val_in_window["val_took_s"].sum()

measured_seconds = train_step_seconds + val_seconds

print("=== MEASURED (iterations 101-200) ===")
print(f"Training-step time:  {train_step_seconds:6.1f}s  (from {len(second_half_blocks)} reported it/sec blocks)")
print(f"Validation-eval time:{val_seconds:6.1f}s  (from {len(val_in_window)} 'Val took' pauses)")
print(f"Total:               {measured_seconds:6.1f}s  ({measured_seconds/60:.2f} min)")
print(f"Avg throughput this half: {second_half_blocks['it_per_sec'].mean():.3f} it/sec, "
      f"{second_half_blocks['tokens_per_sec'].mean():.1f} tokens/sec")
print(f"Peak Metal memory: {train_df['peak_mem_gb'].max():.3f} GB")

print()
print("=== ESTIMATE ONLY -- not a measured figure ===")
est_full_seconds = measured_seconds * 2
print(f"If the first 100 iterations ran at a similar steady-state rate: "
      f"~{est_full_seconds:.0f}s (~{est_full_seconds/60:.1f} min) for the full 200-iteration run,")
print("PLUS an unknown, unmeasured amount for one-time model load / MLX graph")
print("compilation at startup, which this estimate does not include. Treat this")
print("as a ballpark, not the number to cite as 'how long fine-tuning took.'")

=== MEASURED (iterations 101-200) ===
Training-step time:   112.5s  (from 10 reported it/sec blocks)
Validation-eval time:  29.4s  (from 5 'Val took' pauses)
Total:                141.9s  (2.37 min)
Avg throughput this half: 0.889 it/sec, 2093.5 tokens/sec
Peak Metal memory: 8.741 GB

=== ESTIMATE ONLY -- not a measured figure ===
If the first 100 iterations ran at a similar steady-state rate: ~284s (~4.7 min) for the full 200-iteration run,
PLUS an unknown, unmeasured amount for one-time model load / MLX graph
compilation at startup, which this estimate does not include. Treat this
as a ballpark, not the number to cite as 'how long fine-tuning took.'


### Note on re-running this notebook locally

This repo's `.venv/bin/python3` currently symlinks to a pyenv path
(`~/.pyenv/versions/3.12.11/bin/python3`) that wasn't reachable when this
notebook was authored -- if `import pandas` / `import matplotlib` fails when
you run this yourself, check that the venv's Python interpreter still
resolves (`ls -la .venv/bin/python3`) and recreate the venv if the symlink is
broken, rather than assuming a missing-dependency error.